## Training -  stage 1

In [ ]:
%matplotlib inline
import os, torch, pydicom, warnings
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
import SimpleITK as sitk
from skimage import morphology, measure, segmentation

warnings.filterwarnings('ignore')

# ==================== CONFIGURATION ====================
DATA_DIR = "/kaggle/input/datasets/romualdosebany/intertitial-lung-disease/ILD_DB/ILD_DB_volumeROIs"
PATCH_DEPTH, PATCH_HEIGHT, PATCH_WIDTH = 32, 128, 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_EPOCHS = 80
BATCH_SIZE = 4

CLASS_NAMES = {0: "Background", 1: "GGO", 2: "Reticulation", 3: "Consolidation"}
NUM_CLASSES = len(CLASS_NAMES)


# ==================== PREPROCESSING ====================
def resample_to_isotropic(volume, spacing, is_mask=False):
    image = sitk.GetImageFromArray(volume.transpose(2, 0, 1)) 
    image.SetSpacing(spacing)
    new_spacing = [1.0, 1.0, 1.0]
    original_size = image.GetSize()
    original_spacing = image.GetSpacing()
    new_size = [int(round(s * (os / ns))) for s, os, ns in zip(original_size, original_spacing, new_spacing)]
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    resampled_img = resampler.Execute(image)
    return sitk.GetArrayFromImage(resampled_img).transpose(1, 2, 0)

def extract_lung_mask(slice_2d):
    try:
        binary = slice_2d < -400
        cleared = morphology.remove_small_objects(binary, min_size=100)
        cleared = segmentation.clear_border(cleared)
        labels = measure.label(cleared)
        regions = measure.regionprops(labels)
        if not regions: return np.ones_like(slice_2d, dtype=np.float32)
        regions.sort(key=lambda x: x.area, reverse=True)
        mask = np.zeros_like(slice_2d)
        for rg in regions[:2]: mask[labels == rg.label] = 1
        return ndimage.binary_fill_holes(mask).astype(np.float32)
    except: return np.ones_like(slice_2d, dtype=np.float32)

def preprocess_volume(volume):
    masked_vol = np.zeros_like(volume, dtype=np.float32)
    for i in range(volume.shape[2]):
        m = extract_lung_mask(volume[:,:,i])
        masked_vol[:,:,i] = volume[:,:,i] * m
    lower, upper = -1350, 150
    masked_vol = np.clip(masked_vol, lower, upper)
    return ((masked_vol - lower) / (upper - lower)).astype(np.float32)

# ==================== DATA LOADING ====================
def load_patient_data(patient_path):
    patient_id = os.path.basename(patient_path)

    ct_files = sorted([os.path.join(patient_path, f) for f in os.listdir(patient_path) if f.lower().endswith('.dcm')])
    slices, spacing = [], None
    for i, f in enumerate(ct_files):
        try:
            ds = pydicom.dcmread(f)
            if i == 0: spacing = [float(ds.PixelSpacing[0]), float(ds.PixelSpacing[1]), float(getattr(ds, 'SliceThickness', 1.0))]
            img = ds.pixel_array.astype(np.float32)
            if hasattr(ds, 'RescaleSlope'): img = img * ds.RescaleSlope + ds.RescaleIntercept
            slices.append(img)
        except: continue
    
    mask_path = os.path.join(patient_path, 'roi_mask')
    masks = []
    if os.path.exists(mask_path):
        for f in sorted(os.listdir(mask_path)):
            if f.startswith('.'): continue
            try:
                full_m = os.path.join(mask_path, f)
                m = pydicom.dcmread(full_m).pixel_array if f.endswith('.dcm') else np.array(Image.open(full_m).convert('L'))
                # IMPORTANT: Clip values to 0-3 to prevent CUDA DEVICE ASSERT
                m = np.clip(np.round(m), 0, NUM_CLASSES - 1)
                masks.append(m.astype(np.float32))
            except: continue

    if not slices or not masks: return None, None
    min_d = min(len(slices), len(masks))
    vol, mask = np.stack(slices, axis=2)[:,:,:min_d], np.stack(masks, axis=2)[:,:,:min_d]

    if spacing:
        vol = resample_to_isotropic(vol, spacing, is_mask=False)
        mask = resample_to_isotropic(mask, spacing, is_mask=True)
        # Re-verify mask values after interpolation
        mask = np.clip(np.round(mask), 0, NUM_CLASSES - 1)

    if vol.shape[2] < PATCH_DEPTH:
        pad = PATCH_DEPTH - vol.shape[2]
        vol, mask = np.pad(vol, ((0,0), (0,0), (0, pad))), np.pad(mask, ((0,0), (0,0), (0, pad)))

    return preprocess_volume(vol), mask

# ==================== ARCHITECTURE MODEL ====================
class ResidualBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch))
        self.shortcut = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x): return F.relu(self.conv(x) + self.shortcut(x))

class UNet3DDeepSup(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.enc1, self.enc2, self.enc3 = ResidualBlock3D(1, 32), ResidualBlock3D(32, 64), ResidualBlock3D(64, 128)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = ResidualBlock3D(128, 256)
        self.up3, self.dec3, self.ds3 = nn.ConvTranspose3d(256, 128, 2, 2), ResidualBlock3D(256, 128), nn.Conv3d(128, n_classes, 1)
        self.up2, self.dec2, self.ds2 = nn.ConvTranspose3d(128, 64, 2, 2), ResidualBlock3D(128, 64), nn.Conv3d(64, n_classes, 1)
        self.up1, self.dec1, self.final = nn.ConvTranspose3d(64, 32, 2, 2), ResidualBlock3D(64, 32), nn.Conv3d(32, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([F.interpolate(self.up3(b), size=e3.shape[2:]), e3], 1))
        d2 = self.dec2(torch.cat([F.interpolate(self.up2(d3), size=e2.shape[2:]), e2], 1))
        d1 = self.dec1(torch.cat([F.interpolate(self.up1(d2), size=e1.shape[2:]), e1], 1))
        if self.training:
            out_ds2 = F.interpolate(self.ds2(d2), size=d1.shape[2:], mode='trilinear')
            out_ds3 = F.interpolate(self.ds3(d3), size=d1.shape[2:], mode='trilinear')
            return self.final(d1), out_ds2, out_ds3
        return self.final(d1)

# ==================== DATASET & LOSS ====================
class ILDDataset(Dataset):
    def __init__(self, patches, augment=False): self.patches, self.augment = patches, augment
    def __len__(self): return len(self.patches)
    def __getitem__(self, idx):
        v, m = self.patches[idx]
        v, m = v.copy(), m.copy()
        if self.augment and np.random.rand() > 0.5:
            ax = np.random.choice([0, 1, 2]); v, m = np.flip(v, ax).copy(), np.flip(m, ax).copy()
        return torch.FloatTensor(v).unsqueeze(0).permute(0, 3, 1, 2), torch.LongTensor(m.astype(np.int64)).permute(2, 0, 1)

class MultiClassFocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, n_classes=4):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.n_classes = alpha, beta, gamma, n_classes
    def forward(self, inputs, targets):
        inputs = F.softmax(inputs, dim=1)
        # Critical fix: Ensure targets are clamped before one_hot to avoid device asserts
        targets = torch.clamp(targets, 0, self.n_classes - 1)
        targets_one_hot = F.one_hot(targets, self.n_classes).permute(0, 4, 1, 2, 3).float()
        tp = (inputs * targets_one_hot).sum(dim=(0, 2, 3, 4))
        fp = (inputs * (1 - targets_one_hot)).sum(dim=(0, 2, 3, 4))
        fn = ((1 - inputs) * targets_one_hot).sum(dim=(0, 2, 3, 4))
        tversky = (tp + 1e-6) / (tp + self.alpha*fp + self.beta*fn + 1e-6)
        return torch.pow(1 - tversky, self.gamma).mean()

# ==================== MAIN TRAINING ====================
def main():
    all_paths = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) 
                 if os.path.isdir(os.path.join(DATA_DIR, f)) and f != 'HRCT_pilot']
    hrct_dir = os.path.join(DATA_DIR, 'HRCT_pilot')
    if os.path.exists(hrct_dir):
        all_paths += [os.path.join(hrct_dir, f) for f in os.listdir(hrct_dir) if os.path.isdir(os.path.join(hrct_dir, f))]
    
    all_patches = []
    print(f"💎 Filtering and Processing Volumes...")
    for path in tqdm(all_paths):
        vol, mask = load_patient_data(path)
        if vol is not None:
            samples = 60 if 'HRCT' in path else 40
            for _ in range(samples):
                z = np.random.randint(0, vol.shape[2] - 32 + 1)
                y = np.random.randint(0, vol.shape[0] - 128 + 1)
                x = np.random.randint(0, vol.shape[1] - 128 + 1)
                vp, mp = vol[y:y+128, x:x+128, z:z+32], mask[y:y+128, x:x+128, z:z+32]
                if np.sum(mp) > 10 or np.random.rand() > 0.85:
                    all_patches.append((vp, mp))

    train_p, val_p = train_test_split(all_patches, test_size=0.15, random_state=42)
    train_loader = DataLoader(ILDDataset(train_p, augment=True), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(ILDDataset(val_p, augment=False), batch_size=BATCH_SIZE)

    model = UNet3DDeepSup(n_classes=NUM_CLASSES).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    # Background class (0) given lower weight, ILD classes (1-3) given higher weight
    crit_ce = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 15.0, 15.0, 15.0]).to(DEVICE))
    crit_tv = MultiClassFocalTverskyLoss(n_classes=NUM_CLASSES)

    best_dice = 0.0
    for epoch in range(NUM_EPOCHS):
        model.train(); epoch_loss = 0
        for v, m in train_loader:
            v, m = v.to(DEVICE), m.to(DEVICE)
            opt.zero_grad()
            o1, o2, o3 = model(v)
            l1 = crit_ce(o1, m) + 1.5 * crit_tv(o1, m)
            l2 = crit_ce(o2, m) + 1.5 * crit_tv(o2, m)
            l3 = crit_ce(o3, m) + 1.5 * crit_tv(o3, m)
            loss = l1 + 0.5*l2 + 0.25*l3
            loss.backward(); opt.step(); epoch_loss += loss.item()

        model.eval(); v_dice = 0
        with torch.no_grad():
            for v, m in val_loader:
                v, m = v.to(DEVICE), m.to(DEVICE)
                p = model(v).argmax(1)
                dice_batch = 0
                for c in range(1, NUM_CLASSES):
                    p_c, m_c = (p == c), (m == c)
                    dice_batch += (2. * (p_c * m_c).sum() + 1e-6) / (p_c.sum() + m_c.sum() + 1e-6)
                v_dice += dice_batch / (NUM_CLASSES - 1)
        
        avg_dice = (v_dice / len(val_loader)).item()
        print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss/len(train_loader):.4f} | Mean Val Dice: {avg_dice:.4f}")
        
        if avg_dice > best_dice:
            best_dice = avg_dice
            torch.save(model.state_dict(), 'best_multiclass_model.pth')

if __name__ == "__main__":
    main()

💎 Filtering and Processing Volumes...


100%|██████████| 113/113 [02:15<00:00,  1.20s/it]


Epoch 01 | Loss: 2.9570 | Mean Val Dice: 0.2972
Epoch 02 | Loss: 2.8024 | Mean Val Dice: 0.3154
Epoch 03 | Loss: 2.7281 | Mean Val Dice: 0.3255
Epoch 04 | Loss: 2.6517 | Mean Val Dice: 0.2010
Epoch 05 | Loss: 2.6049 | Mean Val Dice: 0.1212
Epoch 06 | Loss: 2.5406 | Mean Val Dice: 0.1228
Epoch 07 | Loss: 2.4407 | Mean Val Dice: 0.1676
Epoch 08 | Loss: 2.3700 | Mean Val Dice: 0.1745
Epoch 09 | Loss: 2.2911 | Mean Val Dice: 0.1816
Epoch 10 | Loss: 2.2189 | Mean Val Dice: 0.2156
Epoch 11 | Loss: 2.1525 | Mean Val Dice: 0.2329
Epoch 12 | Loss: 2.0834 | Mean Val Dice: 0.2493
Epoch 13 | Loss: 2.0116 | Mean Val Dice: 0.3355
Epoch 14 | Loss: 1.9493 | Mean Val Dice: 0.3014
Epoch 15 | Loss: 1.8942 | Mean Val Dice: 0.3681
Epoch 16 | Loss: 1.8451 | Mean Val Dice: 0.3096
Epoch 17 | Loss: 1.8159 | Mean Val Dice: 0.3947
Epoch 18 | Loss: 1.7344 | Mean Val Dice: 0.5447
Epoch 19 | Loss: 1.7311 | Mean Val Dice: 0.3312
Epoch 20 | Loss: 1.6757 | Mean Val Dice: 0.5332
Epoch 21 | Loss: 1.6395 | Mean Val Dice:

KeyboardInterrupt: 

## Number of patches

In [ ]:
import os
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
import SimpleITK as sitk
from skimage import morphology, measure, segmentation
from scipy import ndimage
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ========================= CONFIG =========================
DATA_DIR = "/kaggle/input/datasets/romualdosebany/intertitial-lung-disease/ILD_DB/ILD_DB_volumeROIs"
PATCH_DEPTH, PATCH_HEIGHT, PATCH_WIDTH = 32, 128, 128
NUM_CLASSES = 4

samples_per_volume_hrct = 60
samples_per_volume_normal = 40

# ========================= PREPROCESSING FUNCTIONS =========================
def resample_to_isotropic(volume, spacing, is_mask=False):
    image = sitk.GetImageFromArray(volume.transpose(2, 0, 1))
    image.SetSpacing(spacing)
    new_spacing = [1.0, 1.0, 1.0]
    original_size = image.GetSize()
    original_spacing = image.GetSpacing()
    new_size = [int(round(s * (os / ns))) for s, os, ns in zip(original_size, original_spacing, new_spacing)]
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    resampled_img = resampler.Execute(image)
    return sitk.GetArrayFromImage(resampled_img).transpose(1, 2, 0)

def extract_lung_mask(slice_2d):
    try:
        binary = slice_2d < -400
        cleared = morphology.remove_small_objects(binary, min_size=100)
        cleared = segmentation.clear_border(cleared)
        labels = measure.label(cleared)
        regions = measure.regionprops(labels)
        if not regions:
            return np.ones_like(slice_2d, dtype=np.float32)
        regions.sort(key=lambda x: x.area, reverse=True)
        mask = np.zeros_like(slice_2d)
        for rg in regions[:2]:
            mask[labels == rg.label] = 1
        return ndimage.binary_fill_holes(mask).astype(np.float32)
    except:
        return np.ones_like(slice_2d, dtype=np.float32)

def preprocess_volume(volume):
    masked_vol = np.zeros_like(volume, dtype=np.float32)
    for i in range(volume.shape[2]):
        m = extract_lung_mask(volume[:,:,i])
        masked_vol[:,:,i] = volume[:,:,i] * m
    lower, upper = -1350, 150
    masked_vol = np.clip(masked_vol, lower, upper)
    return ((masked_vol - lower) / (upper - lower)).astype(np.float32)

def load_patient_data(patient_path):
    patient_id = os.path.basename(patient_path)

    # Load CT slices
    ct_files = sorted([os.path.join(patient_path, f) for f in os.listdir(patient_path) if f.lower().endswith('.dcm')])
    slices, spacing = [], None
    for i, f in enumerate(ct_files):
        try:
            ds = pydicom.dcmread(f)
            if i == 0:
                spacing = [float(ds.PixelSpacing[0]), float(ds.PixelSpacing[1]), 
                          float(getattr(ds, 'SliceThickness', 1.0))]
            img = ds.pixel_array.astype(np.float32)
            if hasattr(ds, 'RescaleSlope'):
                img = img * ds.RescaleSlope + ds.RescaleIntercept
            slices.append(img)
        except:
            continue

    # Load masks
    mask_path = os.path.join(patient_path, 'roi_mask')
    masks = []
    if os.path.exists(mask_path):
        for f in sorted(os.listdir(mask_path)):
            if f.startswith('.'): continue
            try:
                full_m = os.path.join(mask_path, f)
                if f.endswith('.dcm'):
                    m = pydicom.dcmread(full_m).pixel_array
                else:
                    m = np.array(Image.open(full_m).convert('L'))
                m = np.clip(np.round(m), 0, NUM_CLASSES - 1)
                masks.append(m.astype(np.float32))
            except:
                continue

    if not slices or not masks:
        return None, None

    min_d = min(len(slices), len(masks))
    vol = np.stack(slices, axis=2)[:, :, :min_d]
    mask = np.stack(masks, axis=2)[:, :, :min_d]

    if spacing:
        vol = resample_to_isotropic(vol, spacing, False)
        mask = resample_to_isotropic(mask, spacing, True)
        mask = np.clip(np.round(mask), 0, NUM_CLASSES - 1)

    if vol.shape[2] < PATCH_DEPTH:
        pad = PATCH_DEPTH - vol.shape[2]
        vol = np.pad(vol, ((0,0),(0,0),(0,pad)))
        mask = np.pad(mask, ((0,0),(0,0),(0,pad)))

    return preprocess_volume(vol), mask


# ========================= PATCH COUNTER =========================
def count_patches():
    all_paths = []
    # Normal patients
    for f in os.listdir(DATA_DIR):
        path = os.path.join(DATA_DIR, f)
        if os.path.isdir(path) and f != 'HRCT_pilot':
            all_paths.append(path)
    
    # HRCT_pilot patients
    hrct_dir = os.path.join(DATA_DIR, 'HRCT_pilot')
    if os.path.exists(hrct_dir):
        for f in os.listdir(hrct_dir):
            path = os.path.join(hrct_dir, f)
            if os.path.isdir(path):
                all_paths.append(path)

    all_patches = []
    print("🔄 Counting patches from all volumes...\n")

    for path in tqdm(all_paths):
        vol, mask = load_patient_data(path)
        if vol is None:
            continue
            
        n_samples = samples_per_volume_hrct if 'HRCT' in path.upper() else samples_per_volume_normal
        
        for _ in range(n_samples):
            if vol.shape[2] < PATCH_DEPTH or vol.shape[0] < PATCH_HEIGHT or vol.shape[1] < PATCH_WIDTH:
                continue
                
            z = np.random.randint(0, vol.shape[2] - PATCH_DEPTH + 1)
            y = np.random.randint(0, vol.shape[0] - PATCH_HEIGHT + 1)
            x = np.random.randint(0, vol.shape[1] - PATCH_WIDTH + 1)
            
            mp = mask[y:y+PATCH_HEIGHT, x:x+PATCH_WIDTH, z:z+PATCH_DEPTH]
            
            # Keep patch if it has enough pathology or randomly
            if np.sum(mp > 0) > 10 or np.random.rand() > 0.85:
                all_patches.append(1)

    total = len(all_patches)
    train_size = int(total * 0.85)   # 85% train, 15% val
    
    print("\n" + "="*70)
    print("📊 FINAL PATCH STATISTICS")
    print("="*70)
    print(f"Total patches generated    : {total:,}")
    print(f"Training patches           : {train_size:,}  ({train_size/total:.1%})")
    print(f"Validation patches         : {total - train_size:,}  ({(total-train_size)/total:.1%})")
    print("="*70)
    
    return total, train_size


# ========================= RUN =========================
if __name__ == "__main__":
    count_patches()

🔄 Counting patches from all volumes...



100%|██████████| 113/113 [02:35<00:00,  1.38s/it]


📊 FINAL PATCH STATISTICS
Total patches generated    : 3,861
Training patches           : 3,281  (85.0%)
Validation patches         : 580  (15.0%)


## Training - Stage 2 

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pydicom
from PIL import Image
import SimpleITK as sitk
from skimage import morphology, measure, segmentation
from scipy import ndimage
from scipy.ndimage import uniform_filter
import warnings
warnings.filterwarnings('ignore')

# =========================================================================
# CONFIGURATION
# =========================================================================
DATA_DIR             = "/kaggle/input/datasets/romualdosebany/intertitial-lung-disease/ILD_DB/ILD_DB_volumeROIs"
PREVIOUS_MODEL_PATH  = "/kaggle/input/datasets/romualdosebany/fine-tuning-unet/best_finetuned_multiclass_model (1).pth"
FINE_TUNED_MODEL_PATH = "best_finetuned_multiclass_model.pth"

PATCH_DEPTH, PATCH_HEIGHT, PATCH_WIDTH = 32, 128, 128
BATCH_SIZE   = 8          # 2x T4: 4 samples per GPU
NUM_EPOCHS   = 60
NUM_CLASSES  = 4
IN_CHANNELS  = 2          # HU intensity + local variance channel

# Per-class patch acceptance thresholds
MIN_GGO_VOXELS       = 200
MIN_RET_VOXELS       = 100
MIN_CONS_VOXELS      = 80
BACKGROUND_KEEP_PROB = 0.08

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =========================================================================
# LOSS FUNCTIONS
# =========================================================================
class GeneralizedDiceLoss(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.n_classes = n_classes

    def forward(self, inputs, targets):
        probs = F.softmax(inputs, dim=1)
        targets_one_hot = (F.one_hot(targets, num_classes=self.n_classes)
                             .permute(0, 4, 1, 2, 3).float())

        counts  = targets_one_hot.sum(dim=(0, 2, 3, 4))
        weights = 1.0 / (counts ** 2 + 1e-6)
        weights = weights / weights.sum()

        gdl = 0.0
        for c in range(self.n_classes):
            p_c          = probs[:, c]
            g_c          = targets_one_hot[:, c]
            intersection = (p_c * g_c).sum()
            union        = p_c.sum() + g_c.sum()
            gdl += weights[c] * (1.0 - (2.0 * intersection + 1e-6) / (union + 1e-6))

        return gdl


class MultiClassFocalTverskyLoss(nn.Module):
    def __init__(self, n_classes=4, alpha=0.3, beta=0.7, gamma=2.0):
        super().__init__()
        self.n_classes = n_classes
        self.alpha = alpha   # FP penalty
        self.beta  = beta    # FN penalty (keep high: missing lesion > false alarm)
        self.gamma = gamma

    def forward(self, inputs, targets):
        probs = F.softmax(inputs, dim=1)
        targets_one_hot = (F.one_hot(targets, num_classes=self.n_classes)
                             .permute(0, 4, 1, 2, 3).float())

        tversky_loss = 0.0
        for c in range(1, self.n_classes):
            p_c          = probs[:, c]
            g_c          = targets_one_hot[:, c]
            tp           = (p_c * g_c).sum()
            fp           = (p_c * (1.0 - g_c)).sum()
            fn           = ((1.0 - p_c) * g_c).sum()
            tversky_idx  = (tp + 1e-6) / (tp + self.alpha * fp + self.beta * fn + 1e-6)
            tversky_loss += (1.0 - tversky_idx) ** (1.0 / self.gamma)

        return tversky_loss / (self.n_classes - 1)

# =========================================================================
# ARCHITECTURE: 3D RESIDUAL U-NET WITH DEEP SUPERVISION
# =========================================================================
class ResidualBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch))
        self.shortcut = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x))


class UNet3DDeepSup(nn.Module):
    def __init__(self, n_classes=4, in_channels=2):
        super().__init__()
        self.enc1      = ResidualBlock3D(in_channels, 32)
        self.enc2      = ResidualBlock3D(32, 64)
        self.enc3      = ResidualBlock3D(64, 128)
        self.pool      = nn.MaxPool3d(2)
        self.bottleneck = ResidualBlock3D(128, 256)

        self.up3  = nn.ConvTranspose3d(256, 128, 2, 2)
        self.dec3 = ResidualBlock3D(256, 128)
        self.ds3  = nn.Conv3d(128, n_classes, 1)

        self.up2  = nn.ConvTranspose3d(128, 64, 2, 2)
        self.dec2 = ResidualBlock3D(128, 64)
        self.ds2  = nn.Conv3d(64, n_classes, 1)

        self.up1  = nn.ConvTranspose3d(64, 32, 2, 2)
        self.dec1 = ResidualBlock3D(64, 32)
        self.final = nn.Conv3d(32, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))

        d3 = self.dec3(torch.cat([F.interpolate(self.up3(b),  size=e3.shape[2:]), e3], 1))
        d2 = self.dec2(torch.cat([F.interpolate(self.up2(d3), size=e2.shape[2:]), e2], 1))
        d1 = self.dec1(torch.cat([F.interpolate(self.up1(d2), size=e1.shape[2:]), e1], 1))

        if self.training:
            out_ds2 = F.interpolate(self.ds2(d2), size=d1.shape[2:],
                                    mode='trilinear', align_corners=False)
            out_ds3 = F.interpolate(self.ds3(d3), size=d1.shape[2:],
                                    mode='trilinear', align_corners=False)
            return self.final(d1), out_ds2, out_ds3
        return self.final(d1)

# =========================================================================
# PREPROCESSING
# =========================================================================
def resample_to_isotropic(volume, spacing, is_mask=False):
    image = sitk.GetImageFromArray(volume.transpose(2, 0, 1))
    image.SetSpacing(spacing)
    new_spacing      = [1.0, 1.0, 1.0]
    original_size    = image.GetSize()
    original_spacing = image.GetSpacing()
    new_size = [int(round(s * (os / ns)))
                for s, os, ns in zip(original_size, original_spacing, new_spacing)]
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    return sitk.GetArrayFromImage(resampler.Execute(image)).transpose(1, 2, 0)


def extract_lung_mask(slice_2d):
    try:
        binary  = slice_2d < -400
        cleared = morphology.remove_small_objects(binary, min_size=100)
        cleared = segmentation.clear_border(cleared)
        labels  = measure.label(cleared)
        regions = measure.regionprops(labels)
        if not regions:
            return np.ones_like(slice_2d, dtype=np.float32)
        regions.sort(key=lambda x: x.area, reverse=True)
        mask = np.zeros_like(slice_2d)
        for rg in regions[:2]:
            mask[labels == rg.label] = 1
        return ndimage.binary_fill_holes(mask).astype(np.float32)
    except:
        return np.ones_like(slice_2d, dtype=np.float32)


def add_variance_channel(vol, radius=3):
    """
    Local HU standard deviation map.
    GGO shows as moderate, non-zero haze — distinct from:
      - Normal lung  (very low variance, near-zero)
      - Consolidation (also near-zero but high absolute HU)
    Gives the encoder an explicit texture cue for GGO.
    """
    kernel  = radius * 2 + 1
    vol_f64 = vol.astype(np.float64)
    mean    = uniform_filter(vol_f64,        size=kernel)
    mean_sq = uniform_filter(vol_f64 ** 2,   size=kernel)
    var     = np.sqrt(np.maximum(mean_sq - mean ** 2, 0)).astype(np.float32)
    v_max   = var.max()
    return var / (v_max + 1e-6)


def preprocess_volume(volume):
    # Lung masking
    masked_vol = np.zeros_like(volume, dtype=np.float32)
    for i in range(volume.shape[2]):
        m = extract_lung_mask(volume[:, :, i])
        masked_vol[:, :, i] = volume[:, :, i] * m

    # HU windowing & normalization
    lower, upper = -1350, 150
    masked_vol   = np.clip(masked_vol, lower, upper)
    hu_norm      = ((masked_vol - lower) / (upper - lower)).astype(np.float32)

    # Variance channel
    variance = add_variance_channel(hu_norm)

    # Stack → (H, W, D, 2)
    return np.stack([hu_norm, variance], axis=-1)


def load_patient_data(patient_path):
    patient_id = os.path.basename(patient_path)

    ct_files = sorted([os.path.join(patient_path, f)
                       for f in os.listdir(patient_path) if f.lower().endswith('.dcm')])
    slices, spacing = [], None

    for i, f in enumerate(ct_files):
        try:
            ds = pydicom.dcmread(f)
            if i == 0:
                spacing = [float(ds.PixelSpacing[0]),
                           float(ds.PixelSpacing[1]),
                           float(getattr(ds, 'SliceThickness', 1.0))]
            img = ds.pixel_array.astype(np.float32)
            if hasattr(ds, 'RescaleSlope'):
                img = img * ds.RescaleSlope + ds.RescaleIntercept
            slices.append(img)
        except:
            continue

    mask_path = os.path.join(patient_path, 'roi_mask')
    masks = []
    if os.path.exists(mask_path):
        for f in sorted(os.listdir(mask_path)):
            if f.startswith('.'):
                continue
            try:
                full_m = os.path.join(mask_path, f)
                m = (pydicom.dcmread(full_m).pixel_array if f.endswith('.dcm')
                     else np.array(Image.open(full_m).convert('L')))
                m = np.clip(np.round(m), 0, NUM_CLASSES - 1)
                masks.append(m.astype(np.float32))
            except:
                continue

    if not slices or not masks:
        return None, None

    min_d = min(len(slices), len(masks))
    vol   = np.stack(slices, axis=2)[:, :, :min_d]
    mask  = np.stack(masks,  axis=2)[:, :, :min_d]

    if spacing:
        vol  = resample_to_isotropic(vol,  spacing, False)
        mask = resample_to_isotropic(mask, spacing, True)
        mask = np.clip(np.round(mask), 0, NUM_CLASSES - 1)

    if vol.shape[2] < PATCH_DEPTH:
        pad  = PATCH_DEPTH - vol.shape[2]
        vol  = np.pad(vol,  ((0, 0), (0, 0), (0, pad)))
        mask = np.pad(mask, ((0, 0), (0, 0), (0, pad)))

    # Returns: vol (H, W, D, 2), mask (H, W, D)
    return preprocess_volume(vol), mask

# =========================================================================
# DATASET
# =========================================================================
class ILDDataset(Dataset):
    def __init__(self, patches, augment=False):
        self.patches = patches
        self.augment = augment

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        v, m = self.patches[idx]
        v, m = v.copy(), m.copy()   # v: (H, W, D, 2)

        if self.augment:
            # Random axis flip
            if np.random.rand() > 0.5:
                ax = np.random.choice([0, 1, 2])
                v  = np.flip(v, ax).copy()
                m  = np.flip(m, ax).copy()

            # Random 90-degree rotation in axial plane
            if np.random.rand() > 0.5:
                k = np.random.randint(1, 4)
                v = np.rot90(v, k, axes=(0, 1)).copy()
                m = np.rot90(m, k, axes=(0, 1)).copy()

            # Intensity jitter on HU channel only
            if np.random.rand() > 0.7:
                v[..., 0] = np.clip(v[..., 0] + np.random.uniform(-0.05, 0.05), 0, 1)

            # Gaussian noise
            if np.random.rand() > 0.8:
                v[..., 0] = np.clip(
                    v[..., 0] + np.random.normal(0, 0.02, v[..., 0].shape), 0, 1
                ).astype(np.float32)

        # (H, W, D, 2) → (2, D, H, W)
        v_tensor = torch.FloatTensor(v).permute(3, 2, 0, 1)
        m_tensor = torch.LongTensor(m.astype(np.int64)).permute(2, 0, 1)
        return v_tensor, m_tensor

# =========================================================================
# TEST-TIME AUGMENTATION
# =========================================================================
def predict_with_tta(model, vol_tensor):
    """
    vol_tensor : (1, 2, D, H, W) on device
    Returns    : (1, D, H, W) predicted label map
    """
    flip_axes = [None, [2], [3], [4], [2, 3]]
    preds = []
    with torch.no_grad():
        for axes in flip_axes:
            v      = vol_tensor.flip(axes) if axes is not None else vol_tensor
            logits = model(v)
            prob   = F.softmax(logits, dim=1)
            prob   = prob.flip(axes) if axes is not None else prob
            preds.append(prob)
    return torch.stack(preds).mean(0).argmax(1)

# =========================================================================
# MULTI-GPU HELPERS
# =========================================================================
def get_model_state(model):
    """Return state dict regardless of DataParallel wrapping."""
    return (model.module.state_dict()
            if isinstance(model, nn.DataParallel)
            else model.state_dict())


def load_state_into_model(model, state_dict):
    """Shape-checked partial load — safe across in_channels change."""
    current = get_model_state(model)
    compatible = {k: v for k, v in state_dict.items()
                  if k in current and current[k].shape == v.shape}
    current.update(compatible)
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(current)
    else:
        model.load_state_dict(current)
    return len(compatible), len(state_dict) - len(compatible)

# =========================================================================
# MAIN
# =========================================================================
def main():
    # ---- GPU setup -------------------------------------------------------
    n_gpus = torch.cuda.device_count()
    print(f"🖥️  Device : {DEVICE}")
    print(f"🖥️  GPUs   : {n_gpus} × {torch.cuda.get_device_name(0) if n_gpus > 0 else 'CPU'}")

    # ---- Data paths ------------------------------------------------------
    all_paths = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR)
                 if os.path.isdir(os.path.join(DATA_DIR, f)) and f != 'HRCT_pilot']
    hrct_dir  = os.path.join(DATA_DIR, 'HRCT_pilot')
    if os.path.exists(hrct_dir):
        all_paths += [os.path.join(hrct_dir, f) for f in os.listdir(hrct_dir)
                      if os.path.isdir(os.path.join(hrct_dir, f))]

    # ---- Patch mining ----------------------------------------------------
    all_patches = []
    print("\n🔥 Mining Focused Lesion-Centered Data Patches...")

    for path in tqdm(all_paths):
        vol, mask = load_patient_data(path)
        if vol is None:
            continue

        ggo_indices  = np.argwhere(mask == 1)
        ret_indices  = np.argwhere(mask == 2)
        cons_indices = np.argwhere(mask == 3)

        n_samples = 120 if 'HRCT' in path else 70

        for _ in range(n_samples):
            rand_val = np.random.rand()

            # Lesion-centered sampling strategy
            if len(ggo_indices) > 0 and rand_val < 0.65:
                c = ggo_indices[np.random.choice(len(ggo_indices))]
            elif len(ret_indices) > 0 and rand_val < 0.85:
                c = ret_indices[np.random.choice(len(ret_indices))]
            elif len(cons_indices) > 0 and rand_val < 0.95:
                c = cons_indices[np.random.choice(len(cons_indices))]
            else:
                c = np.array([np.random.randint(0, vol.shape[0]),
                              np.random.randint(0, vol.shape[1]),
                              np.random.randint(0, vol.shape[2])])

            y_c, x_c, z_c = c[0], c[1], c[2]

            z_start = max(0, min(z_c - PATCH_DEPTH  // 2, vol.shape[2] - PATCH_DEPTH))
            y_start = max(0, min(y_c - PATCH_HEIGHT // 2, vol.shape[0] - PATCH_HEIGHT))
            x_start = max(0, min(x_c - PATCH_WIDTH  // 2, vol.shape[1] - PATCH_WIDTH))

            vp = vol [y_start:y_start+PATCH_HEIGHT,
                      x_start:x_start+PATCH_WIDTH,
                      z_start:z_start+PATCH_DEPTH, :]   # (H, W, D, 2)
            mp = mask[y_start:y_start+PATCH_HEIGHT,
                      x_start:x_start+PATCH_WIDTH,
                      z_start:z_start+PATCH_DEPTH]       # (H, W, D)

            # Per-class voxel acceptance thresholds
            ggo_count  = int(np.sum(mp == 1))
            ret_count  = int(np.sum(mp == 2))
            cons_count = int(np.sum(mp == 3))

            accept = (
                ggo_count  >= MIN_GGO_VOXELS  or
                ret_count  >= MIN_RET_VOXELS   or
                cons_count >= MIN_CONS_VOXELS  or
                np.random.rand() < BACKGROUND_KEEP_PROB
            )
            if accept:
                all_patches.append((vp, mp))

    train_p, val_p = train_test_split(all_patches, test_size=0.12, random_state=42)

    print(f"\n📊 Total Patches Mined : {len(all_patches):,}")
    print(f"   Train              : {len(train_p):,}")
    print(f"   Val                : {len(val_p):,}")

    # ---- DataLoaders -----------------------------------------------------
    # num_workers=4: two workers per GPU keeps the pipeline fed without
    # overwhelming Kaggle's CPU cores
    train_loader = DataLoader(
        ILDDataset(train_p, augment=True),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True)

    val_loader = DataLoader(
        ILDDataset(val_p, augment=False),
        batch_size=BATCH_SIZE,
        num_workers=4, pin_memory=True, persistent_workers=True)

    # ---- Model -----------------------------------------------------------
    model = UNet3DDeepSup(n_classes=NUM_CLASSES, in_channels=IN_CHANNELS).to(DEVICE)

    # Wrap in DataParallel for 2× T4
    if n_gpus > 1:
        print(f"\n🚀 Enabling DataParallel across {n_gpus} GPUs")
        model = nn.DataParallel(model)

    # ---- Warm start ------------------------------------------------------
    if os.path.exists(PREVIOUS_MODEL_PATH):
        print("🔄 WARM START: Loading compatible weights from previous model...")
        try:
            prev_state = torch.load(PREVIOUS_MODEL_PATH, map_location=DEVICE)
            loaded, skipped = load_state_into_model(model, prev_state)
            print(f"   Loaded  : {loaded} layers")
            print(f"   Skipped : {skipped} layers (shape mismatch — expected for enc1 due to in_channels change)")
        except Exception as e:
            print(f"⚠️  Warm start failed: {e}. Starting fresh.")
    else:
        print("⚠️  Previous model not found. Starting fresh.")

    # ---- Optimizer & scheduler -------------------------------------------
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-5, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=4)

    # ---- Loss functions --------------------------------------------------
    # CE weight reduced for GGO: 30→8.
    # GDL + Tversky own the geometric imbalance; stacking CE=30 destabilises
    # gradients for sparse, diffuse GGO voxels.
    crit_ce  = nn.CrossEntropyLoss(
        weight=torch.tensor([0.5, 8.0, 6.0, 4.0]).to(DEVICE))
    crit_gdl = GeneralizedDiceLoss(n_classes=NUM_CLASSES)
    crit_tv  = MultiClassFocalTverskyLoss(n_classes=NUM_CLASSES,
                                          alpha=0.3, beta=0.7, gamma=2.0)

    # ---- Training loop ---------------------------------------------------
    best_dice = 0.0
    print(f"\n{'='*70}")
    print(f"  Starting training — {NUM_EPOCHS} epochs | batch {BATCH_SIZE} | {n_gpus}× T4")
    print(f"{'='*70}\n")

    for epoch in range(NUM_EPOCHS):

        # Ramp deep supervision weights: starts low so the encoder learns
        # from the main output first; auxiliary heads contribute more once
        # the backbone has converged (~epoch 20).
        ds_weight = min(0.4, 0.1 + epoch * 0.015)

        # -- Train ---------------------------------------------------------
        model.train()
        epoch_loss = 0.0

        for v, m in tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS} [train]",
                         leave=False):
            v, m = v.to(DEVICE), m.to(DEVICE)
            optimizer.zero_grad()

            o1, o2, o3 = model(v)

            l1 = crit_ce(o1, m) + 1.00 * crit_tv(o1, m) + 1.00 * crit_gdl(o1, m)
            l2 = crit_ce(o2, m) + 0.50 * crit_tv(o2, m) + 0.50 * crit_gdl(o2, m)
            l3 = crit_ce(o3, m) + 0.25 * crit_tv(o3, m) + 0.25 * crit_gdl(o3, m)

            loss = l1 + ds_weight * l2 + (ds_weight * 0.5) * l3

            loss.backward()
            # Gradient clipping: prevents a rare explosive gradient from a
            # near-empty GGO batch destabilising the run
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)

        # -- Validation with TTA ------------------------------------------
        # TTA is enabled from epoch 0 on validation so the reported Dice
        # is always a realistic proxy for actual inference performance.
        model.eval()
        per_cls = np.zeros(NUM_CLASSES - 1)

        with torch.no_grad():
            for v, m in tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS} [val]  ",
                             leave=False):
                v, m = v.to(DEVICE), m.to(DEVICE)
                pred = predict_with_tta(model, v)

                for ci, c in enumerate(range(1, NUM_CLASSES)):
                    p_c      = (pred == c)
                    m_c      = (m    == c)
                    dice_c   = ((2. * (p_c & m_c).sum() + 1e-6) /
                                (p_c.sum() + m_c.sum() + 1e-6)).item()
                    per_cls[ci] += dice_c

        per_cls  /= len(val_loader)
        avg_dice  = float(per_cls.mean())
        scheduler.step(avg_dice)

        # -- Logging -------------------------------------------------------
        print(
            f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
            f"Loss: {avg_loss:.4f} | "
            f"Dice: {avg_dice:.4f} | "
            f"GGO: {per_cls[0]:.4f} | "
            f"Ret: {per_cls[1]:.4f} | "
            f"Cons: {per_cls[2]:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
            f"DS-w: {ds_weight:.3f}"
        )

        # -- Checkpoint ----------------------------------------------------
        if avg_dice > best_dice:
            best_dice = avg_dice
            # Always save the underlying module state (no DataParallel prefix)
            torch.save(get_model_state(model), FINE_TUNED_MODEL_PATH)
            print(f"   >>> ✅ New best saved — Dice = {best_dice:.4f} <<<")

    print(f"\n{'='*70}")
    print(f"  Training complete.  Peak validation Dice: {best_dice:.4f}")
    print(f"  Model saved to: {FINE_TUNED_MODEL_PATH}")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()

🖥️  Device : cuda
🖥️  GPUs   : 2 × Tesla T4

🔥 Mining Focused Lesion-Centered Data Patches...


100%|██████████| 113/113 [02:16<00:00,  1.21s/it]



📊 Total Patches Mined : 7,890
   Train              : 6,943
   Val                : 947

🚀 Enabling DataParallel across 2 GPUs
🔄 WARM START: Loading compatible weights from previous model...
   Loaded  : 122 layers
   Skipped : 2 layers (shape mismatch — expected for enc1 due to in_channels change)

  Starting training — 60 epochs | batch 8 | 2× T4



Epoch 01/60 | Loss: 2.3511 | Dice: 0.7127 | GGO: 0.5994 | Ret: 0.7991 | Cons: 0.7394 | LR: 8.00e-05 | DS-w: 0.100
   >>> ✅ New best saved — Dice = 0.7127 <<<


Epoch 02/60 | Loss: 1.4966 | Dice: 0.8451 | GGO: 0.8367 | Ret: 0.8924 | Cons: 0.8061 | LR: 8.00e-05 | DS-w: 0.115
   >>> ✅ New best saved — Dice = 0.8451 <<<


...

Epoch 07/60 | Loss: 1.3159 | Dice: 0.9100 | GGO: 0.9407 | Ret: 0.9176 | Cons: 0.8718 | LR: 8.00e-05 | DS-w: 0.190
   >>> ✅ New best saved — Dice = 0.9100 <<<


Epoch 08/60 | Loss: 1.2942 | Dice: 0.9087 | GGO: 0.9393 | Ret: 0.9190 | Cons: 0.8679 | LR: 8.00e-05 | DS-w: 0.205


Epoch 09/60 | Loss: 1.3194 | Dice: 0.9175 | GGO: 0.9501 | Ret: 0.9326 | Cons: 0.8698 | LR: 8.00e-05 | DS-w: 0.220
   >>> ✅ New best saved — Dice = 0.9175 <<<


Epoch 10/60 | Loss: 1.3254 | Dice: 0.9091 | GGO: 0.9253 | Ret: 0.9319 | Cons: 0.8700 | LR: 8.00e-05 | DS-w: 0.235


Epoch 11/60 [train]:   0%|          | 0/868 [00:00<?, ?it/s]

## Distribution metrics

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import pydicom
from PIL import Image
import SimpleITK as sitk
from skimage import morphology, measure, segmentation
from scipy import ndimage
from scipy.ndimage import uniform_filter

# =========================================================================
# CONFIGURATION
# =========================================================================
DATA_DIR              = "/kaggle/input/datasets/romualdosebany/intertitial-lung-disease/ILD_DB/ILD_DB_volumeROIs"
MODEL_PATH            = "/kaggle/input/datasets/romualdosebany/fine-tuning-second/best_finetuned_multiclass_model.pth"

PATCH_DEPTH, PATCH_HEIGHT, PATCH_WIDTH = 32, 128, 128
BATCH_SIZE   = 8  
NUM_CLASSES  = 4
IN_CHANNELS  = 2  

MIN_GGO_VOXELS       = 200
MIN_RET_VOXELS       = 100
MIN_CONS_VOXELS      = 80
BACKGROUND_KEEP_PROB = 0.08

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


CLASS_NAMES = ["Background", "GGO", "Reticular", "Consolidation"]

# =========================================================================
# ARCHITECTURE DEFINITION
# =========================================================================
class ResidualBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch))
        self.shortcut = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x))

class UNet3DDeepSup(nn.Module):
    def __init__(self, n_classes=4, in_channels=2):
        super().__init__()
        self.enc1      = ResidualBlock3D(in_channels, 32)
        self.enc2      = ResidualBlock3D(32, 64)
        self.enc3      = ResidualBlock3D(64, 128)
        self.pool      = nn.MaxPool3d(2)
        self.bottleneck = ResidualBlock3D(128, 256)

        self.up3  = nn.ConvTranspose3d(256, 128, 2, 2)
        self.dec3 = ResidualBlock3D(256, 128)
        self.ds3  = nn.Conv3d(128, n_classes, 1)

        self.up2  = nn.ConvTranspose3d(128, 64, 2, 2)
        self.dec2 = ResidualBlock3D(128, 64)
        self.ds2  = nn.Conv3d(64, n_classes, 1)

        self.up1  = nn.ConvTranspose3d(64, 32, 2, 2)
        self.dec1 = ResidualBlock3D(64, 32)
        self.final = nn.Conv3d(32, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))

        d3 = self.dec3(torch.cat([F.interpolate(self.up3(b),  size=e3.shape[2:]), e3], 1))
        d2 = self.dec2(torch.cat([F.interpolate(self.up2(d3), size=e2.shape[2:]), e2], 1))
        d1 = self.dec1(torch.cat([F.interpolate(self.up1(d2), size=e1.shape[2:]), e1], 1))

        if self.training:
            out_ds2 = F.interpolate(self.ds2(d2), size=d1.shape[2:], mode='trilinear', align_corners=False)
            out_ds3 = F.interpolate(self.ds3(d3), size=d1.shape[2:], mode='trilinear', align_corners=False)
            return self.final(d1), out_ds2, out_ds3
        return self.final(d1)

# =========================================================================
# DATA PREPROCESSING PIPELINE
# =========================================================================
def resample_to_isotropic(volume, spacing, is_mask=False):
    image = sitk.GetImageFromArray(volume.transpose(2, 0, 1))
    image.SetSpacing(spacing)
    new_spacing      = [1.0, 1.0, 1.0]
    original_size    = image.GetSize()
    original_spacing = image.GetSpacing()
    new_size = [int(round(s * (os / ns))) for s, os, ns in zip(original_size, original_spacing, new_spacing)]
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    return sitk.GetArrayFromImage(resampler.Execute(image)).transpose(1, 2, 0)

def extract_lung_mask(slice_2d):
    try:
        binary  = slice_2d < -400
        cleared = morphology.remove_small_objects(binary, min_size=100)
        cleared = segmentation.clear_border(cleared)
        labels  = measure.label(cleared)
        regions = measure.regionprops(labels)
        if not regions: return np.ones_like(slice_2d, dtype=np.float32)
        regions.sort(key=lambda x: x.area, reverse=True)
        mask = np.zeros_like(slice_2d)
        for rg in regions[:2]: mask[labels == rg.label] = 1
        return ndimage.binary_fill_holes(mask).astype(np.float32)
    except:
        return np.ones_like(slice_2d, dtype=np.float32)

def add_variance_channel(vol, radius=3):
    kernel  = radius * 2 + 1
    vol_f64 = vol.astype(np.float64)
    mean    = uniform_filter(vol_f64,       size=kernel)
    mean_sq = uniform_filter(vol_f64 ** 2,   size=kernel)
    var     = np.sqrt(np.maximum(mean_sq - mean ** 2, 0)).astype(np.float32)
    return var / (var.max() + 1e-6)

def preprocess_volume(volume):
    masked_vol = np.zeros_like(volume, dtype=np.float32)
    for i in range(volume.shape[2]):
        m = extract_lung_mask(volume[:, :, i])
        masked_vol[:, :, i] = volume[:, :, i] * m
    lower, upper = -1350, 150
    masked_vol   = np.clip(masked_vol, lower, upper)
    hu_norm      = ((masked_vol - lower) / (upper - lower)).astype(np.float32)
    variance = add_variance_channel(hu_norm)
    return np.stack([hu_norm, variance], axis=-1)

def load_patient_data(patient_path):
    patient_id = os.path.basename(patient_path)
    ct_files = sorted([os.path.join(patient_path, f) for f in os.listdir(patient_path) if f.lower().endswith('.dcm')])
    slices, spacing = [], None
    for i, f in enumerate(ct_files):
        try:
            ds = pydicom.dcmread(f)
            if i == 0: spacing = [float(ds.PixelSpacing[0]), float(ds.PixelSpacing[1]), float(getattr(ds, 'SliceThickness', 1.0))]
            img = ds.pixel_array.astype(np.float32)
            if hasattr(ds, 'RescaleSlope'): img = img * ds.RescaleSlope + ds.RescaleIntercept
            slices.append(img)
        except: continue
    mask_path = os.path.join(patient_path, 'roi_mask')
    masks = []
    if os.path.exists(mask_path):
        for f in sorted(os.listdir(mask_path)):
            if f.startswith('.'): continue
            try:
                full_m = os.path.join(mask_path, f)
                m = pydicom.dcmread(full_m).pixel_array if f.endswith('.dcm') else np.array(Image.open(full_m).convert('L'))
                masks.append(np.clip(np.round(m), 0, NUM_CLASSES - 1).astype(np.float32))
            except: continue
    if not slices or not masks: return None, None, None
    min_d = min(len(slices), len(masks))
    vol = np.stack(slices, axis=2)[:, :, :min_d]
    mask = np.stack(masks,  axis=2)[:, :, :min_d]
    if spacing:
        vol  = resample_to_isotropic(vol,  spacing, False)
        mask = resample_to_isotropic(mask, spacing, True)
        mask = np.clip(np.round(mask), 0, NUM_CLASSES - 1)
    if vol.shape[2] < PATCH_DEPTH:
        pad = PATCH_DEPTH - vol.shape[2]
        vol  = np.pad(vol,  ((0, 0), (0, 0), (0, pad)))
        mask = np.pad(mask, ((0, 0), (0, 0), (0, pad)))
    return preprocess_volume(vol), mask, patient_id

class ILDEvalDataset(torch.utils.data.Dataset):
    def __init__(self, patches): self.patches = patches
    def __len__(self): return len(self.patches)
    def __getitem__(self, idx):
        v, m, pid = self.patches[idx]
        v_tensor = torch.FloatTensor(v.copy()).permute(3, 2, 0, 1)
        m_tensor = torch.LongTensor(m.copy().astype(np.int64)).permute(2, 0, 1)
        return v_tensor, m_tensor, pid

def predict_with_tta(model, vol_tensor):
    flip_axes = [None, [2], [3], [4], [2, 3]]
    preds = []
    with torch.no_grad():
        for axes in flip_axes:
            v      = vol_tensor.flip(axes) if axes is not None else vol_tensor
            logits = model(v)
            prob   = F.softmax(logits, dim=1)
            prob   = prob.flip(axes) if axes is not None else prob
            preds.append(prob)
    return torch.stack(preds).mean(0).argmax(1)

# =========================================================================
# LESION-WISE SENSITIVITY CALCULATION (Connected Components Engine)
# =========================================================================
def compute_lesion_wise_sensitivity(y_true_vol, y_pred_vol, class_idx, min_lesion_size=20, IoU_threshold=0.1):
    true_binary = (y_true_vol == class_idx)
    pred_binary = (y_pred_vol == class_idx)
    
    true_labels, num_true = measure.label(true_binary, return_num=True, connectivity=3)
    if num_true == 0:
        return 0, 0
        
    true_lesions_count = 0
    detected_lesions_count = 0
    
    for label in range(1, num_true + 1):
        lesion_mask = (true_labels == label)
        if np.sum(lesion_mask) < min_lesion_size:
            continue
            
        true_lesions_count += 1
        overlap = np.sum(lesion_mask & pred_binary)
        union = np.sum(lesion_mask | (pred_binary & lesion_mask))
        
        iou = overlap / (union + 1e-6)
        if iou >= IoU_threshold:
            detected_lesions_count += 1
            
    return detected_lesions_count, true_lesions_count

# =========================================================================
# MAIN EVALUATION EXECUTION
# =========================================================================
def main():
    print("🔄 Step 1: Re-mining validation patches with Patient-ID links...")
    all_paths = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f)) and f != 'HRCT_pilot']
    hrct_dir  = os.path.join(DATA_DIR, 'HRCT_pilot')
    if os.path.exists(hrct_dir):
        all_paths += [os.path.join(hrct_dir, f) for f in os.listdir(hrct_dir) if os.path.isdir(os.path.join(hrct_dir, f))]

    all_patches = []
    np.random.seed(42)  
    
    for path in tqdm(all_paths, desc="Processing Volumes"):
        vol, mask, pid = load_patient_data(path)
        if vol is None: continue
        ggo_indices, ret_indices, cons_indices = np.argwhere(mask == 1), np.argwhere(mask == 2), np.argwhere(mask == 3)
        n_samples = 120 if 'HRCT' in path else 70

        for _ in range(n_samples):
            rand_val = np.random.rand()
            if len(ggo_indices) > 0 and rand_val < 0.65: c = ggo_indices[np.random.choice(len(ggo_indices))]
            elif len(ret_indices) > 0 and rand_val < 0.85: c = ret_indices[np.random.choice(len(ret_indices))]
            elif len(cons_indices) > 0 and rand_val < 0.95: c = cons_indices[np.random.choice(len(cons_indices))]
            else: c = np.array([np.random.randint(0, vol.shape[0]), np.random.randint(0, vol.shape[1]), np.random.randint(0, vol.shape[2])])

            y_c, x_c, z_c = c[0], c[1], c[2]
            z_start = max(0, min(z_c - PATCH_DEPTH  // 2, vol.shape[2] - PATCH_DEPTH))
            y_start = max(0, min(y_c - PATCH_HEIGHT // 2, vol.shape[0] - PATCH_HEIGHT))
            x_start = max(0, min(x_c - PATCH_WIDTH  // 2, vol.shape[1] - PATCH_WIDTH))

            vp = vol [y_start:y_start+PATCH_HEIGHT, x_start:x_start+PATCH_WIDTH, z_start:z_start+PATCH_DEPTH, :]
            mp = mask[y_start:y_start+PATCH_HEIGHT, x_start:x_start+PATCH_WIDTH, z_start:z_start+PATCH_DEPTH]

            if int(np.sum(mp == 1)) >= MIN_GGO_VOXELS or int(np.sum(mp == 2)) >= MIN_RET_VOXELS or int(np.sum(mp == 3)) >= MIN_CONS_VOXELS or np.random.rand() < BACKGROUND_KEEP_PROB:
                all_patches.append((vp, mp, pid))

    _, val_p = train_test_split(all_patches, test_size=0.12, random_state=42)
    val_loader = DataLoader(ILDEvalDataset(val_p), batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)

    model = UNet3DDeepSup(n_classes=NUM_CLASSES, in_channels=IN_CHANNELS).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    patient_data = {}
    
    # Global tracking matrix for the entire voxel-level confusion matrix
    global_total_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    print("\n🧠 Running Inference & aggregating predictions back to patient level...")
    with torch.no_grad():
        for v, m, pids in tqdm(val_loader, desc="Inference"):
            v = v.to(DEVICE)
            pred = predict_with_tta(model, v)
            
            pred = pred.cpu().numpy()
            m = m.numpy()
            
            # Update the global voxel-level matrix safely row-by-row
            batch_cm = confusion_matrix(m.ravel(), pred.ravel(), labels=list(range(NUM_CLASSES)))
            global_total_cm += batch_cm
            
            for b in range(len(pids)):
                pid = pids[b]
                if pid not in patient_data:
                    patient_data[pid] = {'true': [], 'pred': []}
                patient_data[pid]['true'].append(m[b])
                patient_data[pid]['pred'].append(pred[b])

    patient_dice_scores = {1: [], 2: [], 3: []} 
    lesion_totals = {1: [0,0], 2: [0,0], 3: [0,0]} 

    print("\n🔬 Processing patient-level distribution metrics & running 3D object parsing...")
    for pid, sub_volumes in patient_data.items():
        y_true_patient = np.concatenate(sub_volumes['true'], axis=0)
        y_pred_patient = np.concatenate(sub_volumes['pred'], axis=0)
        
        for c in range(1, NUM_CLASSES):
            p_c = (y_pred_patient == c)
            m_c = (y_true_patient == c)
            
            if m_c.sum() == 0 and p_c.sum() == 0:
                continue
                
            dice = (2. * (p_c & m_c).sum() + 1e-6) / (p_c.sum() + m_c.sum() + 1e-6)
            patient_dice_scores[c].append(dice)
            
            det, total = compute_lesion_wise_sensitivity(y_true_patient, y_pred_patient, class_idx=c)
            lesion_totals[c][0] += det
            lesion_totals[c][1] += total

    # Normalize Confusion Matrix Percentages
    global_cm_norm = global_total_cm.astype('float') / (global_total_cm.sum(axis=1)[:, np.newaxis] + 1e-9) * 100

    # =========================================================================
    # PRINT RESULTS WITH INTEGRATED CONFUSION MATRIX
    # =========================================================================
    print("\n" + "="*85)
    print(" 1. RAW VOXEL-LEVEL CONFUSION MATRIX")
    print("="*85)
    header = f"{'True \\ Pred':<15} |" + "".join([f"{name:>16}" for name in CLASS_NAMES])
    print(header)
    print("-" * 85)
    for i, row_label in enumerate(CLASS_NAMES):
        row_str = f"{row_label:<15} |" + "".join([f"{global_total_cm[i, j]:>16,}" for j in range(NUM_CLASSES)])
        print(row_str)

    print("\n" + "="*85)
    print(" 2. NORMALIZED CONFUSION MATRIX (%)")
    print("="*85)
    print(header)
    print("-" * 85)
    for i, row_label in enumerate(CLASS_NAMES):
        row_str = f"{row_label:<15} |" + "".join([f"{global_cm_norm[i, j]:>15.2f}%" for j in range(NUM_CLASSES)])
        print(row_str)

    print("\n" + "="*85)
    print(" 3. PATIENT-LEVEL DICE PERFORMANCE DISTRIBUTION (Mean ± Std)")
    print("="*85)
    for c in range(1, NUM_CLASSES):
        scores = np.array(patient_dice_scores[c])
        if len(scores) > 0:
            print(f" Class {CLASS_NAMES[c]:<15} : Mean Dice = {scores.mean():.4f} | Std Dev = ±{scores.std():.4f}  (n={len(scores)} patients)")
        else:
            print(f" Class {CLASS_NAMES[c]:<15} : No active targets found inside validation splits.")

    print("\n" + "="*85)
    print(" 4. OBJECT-LEVEL LESION-WISE SENSITIVITY (3D Connected Components)")
    print("="*85)
    print(f" {'Pathology Class':<18} | {'Detected Lesions':<18} | {'Total Real Lesions':<18} | {'Sensitivity (%)':<15}")
    print("-" * 85)
    for c in range(1, NUM_CLASSES):
        det, total = lesion_totals[c][0], lesion_totals[c][1]
        sens = (det / total * 100) if total > 0 else 0.0
        print(f" {CLASS_NAMES[c]:<18} | {det:<18,} | {total:<18,} | {sens:.2f}%")
    print("="*85)

if __name__ == "__main__":
    main()

🔄 Step 1: Re-mining validation patches with Patient-ID links...


Processing Volumes: 100%|██████████| 113/113 [02:26<00:00,  1.29s/it]



🧠 Running Inference & aggregating predictions back to patient level...


Inference: 100%|██████████| 119/119 [08:14<00:00,  4.16s/it]



🔬 Processing patient-level distribution metrics & running 3D object parsing...

 1. RAW VOXEL-LEVEL CONFUSION MATRIX
True \ Pred     |      Background             GGO       Reticular   Consolidation
-------------------------------------------------------------------------------------
Background      |     478,143,144         289,105         584,488       2,902,061
GGO             |          61,858       3,107,628               0              10
Reticular       |          16,109              29       1,148,637              28
Consolidation   |         316,886              19             233      11,503,365

 2. NORMALIZED CONFUSION MATRIX (%)
True \ Pred     |      Background             GGO       Reticular   Consolidation
-------------------------------------------------------------------------------------
Background      |          99.22%           0.06%           0.12%           0.60%
GGO             |           1.95%          98.05%           0.00%           0.00%
Reticular       |